***

Preparing Workspace

***

In [1]:


## Indicators:
# Production_1
# Production_4
# Production_7
# Location_1
# Location_2a
# Location_2b


import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
import sys
sys.path.append(str(Path.cwd().parent.parent / 'config'))
import functions as func


# SharePoint OneDrive paths
PATH_SERVER = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")
PATH_MAIN = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' / 'Data'
FILE_ABOUT = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' / 'Process Revamp' / 'Task 6. Process Map' / 'About Indicators.xlsx'
# FILE_HOUSING = PATH_MAIN / 'Vibrant and Inclusive Places' / 'Development' / 'SACOG Housing Dataset' /  'Region_SACOG_Housing_INV_Merged_with_About.xlsx' # Developed by Warren
FILE_HOUSING = PATH_MAIN / 'Vibrant and Inclusive Places' / 'Development' / 'SACOG Housing Dataset' / 'Housing_Inventory_2026' / 'SACOG Housing Inventory 2026.xlsx'



def export_housing(df, params, path_out):   
    workbook_name = f'{params['indicator']} {params['geo']} {params['source']}.xlsx'
    file_out = path_out / workbook_name
    if params['indicator'] in ['Production_1', 'Production_6', 'Production_7', 'Location_1', 'Location_2a', 'Location_2b', 'Production_4']:
        if params['indicator'] in ['Production_1', 'Production_6', 'Production_7', 'Production_4']:
            if params['geo'] == 'MPO':
                params['geo'] = 'Six-County Sacramento Region'
        if params['indicator'] in ['Location_1', 'Location_2a', 'Location_2b']:
            if params['geo'] == 'MPO':
                params['geo'] = 'Six-County SACOG Planning Area'
    df_about = func.write_about(params)
    if params['indicator'] in ['Production_1', 'Production_6', 'Production_7', 'Location_1', 'Location_2a', 'Location_2b']:
        if params['indicator'] in ['Production_1', 'Production_6', 'Production_7', 'Production_4']:
            if params['geo'] == 'Six-County Sacramento Region':
                params['geo'] = 'MPO'
        if params['indicator'] in ['Location_1', 'Location_2a', 'Location_2b']:
            if params['geo'] == 'Six-County SACOG Planning Area':
                params['geo'] = 'MPO'
    
    if file_out.exists():
        with pd.ExcelWriter(file_out, mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
            df_about.to_excel(writer, sheet_name='About'      , index=False, header=False)
            df      .to_excel(writer, sheet_name=params['geo'], index=False,             )
    else:
        with pd.ExcelWriter(file_out, engine='xlsxwriter') as writer:
            df_about.to_excel(writer, sheet_name='About'      , index=False, header=False)
            df      .to_excel(writer, sheet_name=params['geo'], index=False,             )



pd.set_option('display.max_columns', None)


EXPORT=True



In [3]:
year_start = 2001
year_end   = 2025

years_to_import = range(year_start, year_end+1)

list_df = []
 

for year in tqdm(years_to_import):
    
    sheet_name=str(year)
    df_year = pd.read_excel(FILE_HOUSING, sheet_name=sheet_name)
    df_year['Year'] = year
    list_df.append(df_year)

df_housing = pd.concat(list_df)
df_housing = df_housing.set_index(['County', 'Jurisdiction', 'Year']).reset_index()
df_housing.loc[df_housing['Jurisdiction'].str.contains('COUNTY'), 'Jurisdiction'] = 'UNINCORPORATED'
df_housing = df_housing[df_housing['County'] != 'Region']
df_housing['MPO'] = 'SACOG'
df_housing = df_housing.set_index(['MPO', 'County', 'Jurisdiction', 'Year']).reset_index()
df_housing = df_housing.sort_values(['MPO', 'County', 'Jurisdiction', 'Year'], ascending = [True, True, True, False])
df_housing = df_housing[df_housing['Jurisdiction']!='Region Total'].reset_index()
df_housing


100%|██████████| 25/25 [00:01<00:00, 12.99it/s]


,index,MPO,County,Jurisdiction,Year,Total,SF_total,MF_total,ADU_total,SF_RR,SF_SFLL,SF_SFSL,MF_2to4,MF_5plus,ADU_RR,ADU_SFLL,ADU_SFSL,SF_DEMO,ADU_DEMO,COMTYP_CC,COMTYP_EC,COMTYP_DC,COMTYP_RR,COMTYP_AGNL,COMTYP_NA,COMTYP_SLT,COMTYP_NO_COMTYP_PRI,GRZ_TOT,GRZ_SF,GRZ_MF2t4,GRZ_MF5,GRZ_NO,GRZ_SF_NO,GRZ_MF2t4_NO,GRZ_MF5_NO,GRZ_SHR,GRZ_RR,GRZ_SFLL,GRZ_SFSL,GRZ_RR_SFLL,GRZ_RR_NO,GRZ_SFLL_NO,GRZ_SFSL_NO,GRZ_RR_SFLL_NO
0,697,SACOG,EL DORADO,PLACERVILLE,2025,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000,0,0,0,0,0,0,0,0
1,668,SACOG,EL DORADO,PLACERVILLE,2024,32,20,2,10,0,18,2,2,0,3,6,1,0,0,0,32,0,0,0,0,0,0,0,0,0,0,32,30,2,0,0.000000,0,0,0,0,3,24,3,27
2,639,SACOG,EL DORADO,PLACERVILLE,2023,4,1,0,3,1,0,0,0,0,0,2,1,2,0,0,4,0,0,0,0,0,0,0,0,0,0,4,4,0,0,0.000000,0,0,0,0,1,2,1,3
3,610,SACOG,EL DORADO,PLACERVILLE,2022,13,3,0,10,-1,4,0,0,0,3,7,0,0,0,0,13,0,0,0,0,0,0,0,0,0,0,13,13,0,0,0.000000,0,0,0,0,2,11,0,13
4,581,SACOG,EL DORADO,PLACERVILLE,2021,15,8,0,7,3,5,0,0,0,0,7,0,1,0,2,13,0,0,0,0,0,0,0,0,0,0,15,15,0,0,0.000000,0,0,0,0,3,12,0,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
720,143,SACOG,YUBA,WHEATLAND,2005,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000,0,0,0,0,0,0,0,0
721,114,SACOG,YUBA,WHEATLAND,2004,278,278,0,0,2,258,18,0,0,0,0,0,0,0,0,0,0,0,0,0,0,278,4,4,0,0,274,274,0,0,0.014388,0,4,0,4,2,254,18,256
722,85,SACOG,YUBA,WHEATLAND,2003,139,139,0,0,0,139,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,139,1,1,0,0,138,138,0,0,0.007194,0,1,0,1,0,138,0,138
723,56,SACOG,YUBA,WHEATLAND,2002,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.000000,0,0,0,0,0,0,0,0


In [ ]:


## Production_1


print('Organizing indicator Production_1 by Jurisdictions')
df_prod1_a = df_housing.copy()
df_prod1_a = df_prod1_a[['MPO', 'County', 'Jurisdiction', 'Year', 'Total']]
display(df_prod1_a.head(5))

print('Organizing indicator Production_1 by Counties')
df_prod1_b = df_housing.copy()
df_prod1_b = df_prod1_b.groupby(['MPO', 'County', 'Year'], as_index = False, sort = False)['Total'].agg('sum')
display(df_prod1_b.head(5))

print('Organizing indicator Production_1 by the entire SACOG Region')
df_prod1_c = df_housing.copy()
df_prod1_c = df_prod1_c.groupby(['MPO', 'Year'], as_index = False, sort = False)['Total'].agg('sum')
display(df_prod1_c.head(5))


df_plot = df_prod1_a.copy()

df_plot = pd.melt(df_plot, id_vars = ['Year', 'County', 'Jurisdiction'])
fig = px.line(df_plot, x='Year', y='value', color='County', line_dash='Jurisdiction', markers=True)
fig.update_layout(title = 'Total Housing Permits by County/Jurisdiction')

fig.show()


df_plot = df_prod1_b.copy()

df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year', 'County'])
fig = px.line(df_plot, x='Year', y='value', color='County', markers=True)
fig.update_layout(title = 'Total Housing Permits by County')

display(df_prod1_b.head())

fig.show()


df_plot = df_prod1_c.copy()

df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year']) 
fig = px.line(df_plot, x='Year', y='value', markers=True)
fig.update_layout(title = 'Total Housing Permits SACOG')

display(df_prod1_c.head())

fig.show()




year_start = df_prod1_c['Year'].min()
year_end   = df_prod1_c['Year'].max()

if EXPORT:
    
    indicator = 'Production_1'
    source = 'SACOG Housing Permit Data'
    sample_type = 'SACOG Housing'
    params = {
        'sample': sample_type
        , 'indicator': indicator
        , 'start_year': year_start
        , 'end_year': year_end
        , 'estimate': None
        , 'moe_thresh': None
        , 'source': source
        , 'geo': 'Sacramento Six-County Region'
    }

    # Update overall about documentations workbook
    df_about = func.write_about(params)
    with pd.ExcelWriter(FILE_ABOUT, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_about.to_excel(writer, index=False, sheet_name=indicator, header=False)


    # Exporting
    print('Exporting...\n')
    
    paths_out = [PATH_MAIN / 'Vibrant and Inclusive Places' / 'Development' / 'Housing Production' / indicator, PATH_SERVER]

    for path_out in paths_out:
        
        params['geo'] = 'Jurisdictions'
        export_housing(df_prod1_a, params, path_out)

        params['geo'] = 'Counties'
        export_housing(df_prod1_b, params, path_out)
        
        params['geo'] = 'MPO'
        export_housing(df_prod1_c, params, path_out)

    print('Success!!')



In [ ]:


## Production_4

# Use pd.read_excel to import the Pop_5 jurisdiction data
# Calculate population growth by year using df_pop_5
# Calculate housing growth by year using df_housing "Total" column
# Merge the two files together by county/jurisdiction
# Roll up to Jurisdiction, County, and MPO levels (3 different data frames)
# Make plots


indicator_name = 'Production_4'

#Importing Datasets 
file_pop5 = PATH_MAIN / 'Vibrant and Inclusive Places' / 'People and Community' / 'Pop and Demographics' / 'Pop_5 CA Regions' / 'Pop_5 DOF Jurisdictions.xlsx'
sheet_name = 'Data'
df_pop_5 = pd.read_excel(file_pop5, sheet_name=sheet_name)
df_pop_5 = df_pop_5.sort_values(['County', 'Jurisdiction', 'Year'], ascending=[True,True,True])
df_pop_5['Population Growth'] = df_pop_5.groupby(['County', 'Jurisdiction'])['Population'].diff()
df_pop_5['County'] = df_pop_5['County'].str.upper() 
df_pop_5['Jurisdiction'] = df_pop_5['Jurisdiction'].str.upper() 

#merging the two files together bby county and jurisdiction
df_merged = pd.merge(df_housing, df_pop_5, on=['MPO', 'County', 'Jurisdiction', 'Year'])
df_merged = df_merged.rename(columns={'Total': 'Number of New Housing Units Built'})

#Jurisdiction level
print('Organizing indicator Production_4 by Jurisdictions')
df_prod4_a = df_merged.copy()
df_prod4_a = df_prod4_a[['MPO', 'County', 'Jurisdiction', 'Year', 'Number of New Housing Units Built', 'Population Growth']]
display(df_prod4_a.head(5))

#County level 
print('Organizing indicator Production_4 by Counties')
df_prod4_b = df_merged.copy()
df_prod4_b = df_prod4_b.groupby(['MPO', 'County', 'Year'], as_index=False, sort=False).agg({'Number of New Housing Units Built': 'sum', 'Population Growth': 'sum'})
display(df_prod4_b.head(5))

#MPO level
print('Organizing indicator Production_4 by MPO')
df_prod4_c = df_merged.copy()
df_prod4_c = df_prod4_c.groupby(['MPO', 'Year'], as_index=False, sort=False).agg({'Number of New Housing Units Built': 'sum', 'Population Growth': 'sum'})
display(df_prod4_c.head(5))


df_plot = df_prod4_a.copy()

df_plot = pd.melt(df_plot, id_vars = ['MPO', 'County', 'Jurisdiction', 'Year'])
fig = px.line(df_plot, x='Year', y='value', color='County', line_dash = 'Jurisdiction', facet_col = 'variable', markers=True)
fig.update_layout(title = 'Total Housing/Population Growth by Jurisdiction')

fig.show()

df_plot = df_prod4_b.copy()

df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year', 'County']) 
fig = px.line(df_plot, x='Year', y='value', color='County', line_dash='variable', markers=True)
fig.update_layout(title = 'Total Housing/Population Growth by County')

fig.show()

df_plot = df_prod4_c.copy()

df_plot['Estimated Number of Households Added'] = df_plot['Population Growth']/2.5

df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year'])
fig = px.line(df_plot[df_plot['variable'] != 'Estimated Number of Households Added'], x='Year', y='value', color='variable', markers=True)
fig.add_trace(go.Scatter(x=df_plot["Year"], y=df_plot[df_plot['variable'] == 'Estimated Number of Households Added']['value']
                         , name = 'Estimated Number of Households Added'
                         , line=go.scatter.Line(color="gray", dash="dot")
                        ))
fig.update_layout(title = 'Total Housing/Population Growth SACOG')

df_plot = df_plot.sort_values(['MPO', 'Year', 'variable'], ascending = [True, False, False])
print(df_plot.variable.unique())
display(df_plot.head())

fig.show()




year_start = df_prod4_c['Year'].min()
year_end   = df_prod4_c['Year'].max()

if EXPORT:

    indicator = 'Production_4'
    source = 'SACOG Housing and DOF Population Data'
    sample_type = 'SACOG Housing'
    params = {
        'sample': sample_type
        , 'indicator': indicator
        , 'start_year': year_start
        , 'end_year': year_end
        , 'estimate': None
        , 'moe_thresh': None
        , 'source': source
        , 'geo': 'Sacramento Six-County Region'
    }

    # Update overall about documentations workbook
    df_about = func.write_about(params)
    with pd.ExcelWriter(FILE_ABOUT, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_about.to_excel(writer, index=False, sheet_name=indicator, header=False)

    # Exporting
    print('Exporting...')
    print('')
    paths_out = [PATH_MAIN / 'Vibrant and Inclusive Places' / 'Development' / 'Housing Production' / indicator, PATH_SERVER]

    for path_out in paths_out:
        
        params['geo'] = 'Jurisdictions'
        export_housing(df_prod4_a, params, path_out)
            
        params['geo'] = 'Counties'
        export_housing(df_prod4_b, params, path_out)
        
        params['geo'] = 'MPO'
        export_housing(df_prod4_c, params, path_out)

    print('Success!!')






In [4]:


## Production_6


gb_jurisdiction = df_housing.groupby(['MPO', 'County', 'Jurisdiction', 'Year'], sort=False, as_index=False)
gb_county       = df_housing.groupby(['MPO', 'County'                , 'Year'], sort=False, as_index=False)
gb_mpo          = df_housing.groupby(['MPO'                          , 'Year'], sort=False, as_index=False)

metrics = ['SF_total', 'MF_total', 'ADU_total', 'SF_RR', 'SF_SFLL', 'SF_SFSL', 'MF_2to4', 'MF_5plus', 'ADU_RR', 'ADU_SFLL', 'ADU_SFSL']

#Jurisdiction level
print('Organizing indicator Production_6 by Jurisdictions')
df_prod6_a = gb_jurisdiction[metrics].sum() 
display(df_prod6_a.head(5))

#County level 
print('Organizing indicator Production_6 by Counties')
df_prod6_b = gb_county[metrics].sum()
display(df_prod6_b.head(5))

#MPO level
print('Organizing indicator Production_6 by MPO')
df_prod6_c = gb_mpo[metrics].sum()
display(df_prod6_c.head(5))


df_plota = df_prod6_a.copy()
df_plotb = df_prod6_a.copy()
df_plot1 = df_prod6_a.copy()
df_plot2 = df_prod6_a.copy()
df_plot3 = df_prod6_a.copy()
df_plot4 = df_prod6_a.copy()
df_plot5 = df_prod6_a.copy()


df_plota = pd.melt(df_plota, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plota = df_plota[df_plota['variable'] == 'MF_total']
fig = px.line(df_plota, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'MF Units by Jurisdiction')
fig.show()

df_plotb = pd.melt(df_plotb, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plotb = df_plotb[df_plotb['variable'] == 'SF_total']
fig = px.line(df_plotb, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'SF Units by Jurisdiction')
fig.show()

df_plot1 = pd.melt(df_plot1, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plot1 = df_plot1[df_plot1['variable'] == 'MF_2to4']
fig = px.line(df_plot1, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'MF 2 to 4 Units by Jurisdiction')
fig.show()

df_plot2 = pd.melt(df_plot2, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plot2 = df_plot2[df_plot2['variable'] == 'MF_5plus']
fig = px.line(df_plot2, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'MF 5+ Units by Jurisdiction')
fig.show()

df_plot3 = pd.melt(df_plot3, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plot3 = df_plot3[df_plot3['variable'] == 'SF_RR']
fig = px.line(df_plot3, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'SF Rural Residential Units by Jurisdiction')
fig.show()

df_plot4 = pd.melt(df_plot4, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plot4 = df_plot4[df_plot4['variable'] == 'SF_SFLL']
fig = px.line(df_plot4, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'SF Large Lot Units by Jurisdiction')
fig.show()

df_plot5 = pd.melt(df_plot5, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plot5 = df_plot5[df_plot5['variable'] == 'SF_SFSL']
fig = px.line(df_plot5, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'SF Small Lot Units by Jurisdiction')
fig.show()


df_plota = df_prod6_b.copy()
df_plot1 = df_prod6_b.copy()
df_plot3 = df_prod6_b.copy()


df_plota = pd.melt(df_plota, id_vars = ['Year', 'County'])
df_plota = df_plota[df_plota['variable'].isin(['MF_total', 'SF_total'])]
fig = px.line(df_plota, x='Year', y='value', color='County', line_dash = 'variable', markers=True)
fig.update_layout(title = 'MF vs SF Units by County')
fig.show()

df_plot1 = pd.melt(df_plot1, id_vars = ['Year', 'County'])
df_plot1 = df_plot1[df_plot1['variable'].isin(['MF_2to4', 'MF_5plus'])]
fig = px.line(df_plot1, x='Year', y='value', color='County', line_dash = 'variable', markers=True)
fig.update_layout(title = 'MF Units by County')
fig.show()

df_plot3 = pd.melt(df_plot3, id_vars = ['Year', 'County'])
df_plot3 = df_plot3[df_plot3['variable'].isin(['SF_RR', 'SF_SFLL', 'SF_SFSL'])]
fig = px.line(df_plot3, x='Year', y='value', color='County', line_dash = 'variable', markers=True)
fig.update_layout(title = 'SF Units by County')
fig.show()



df_plot = df_prod6_c.copy()
df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year'])
df_plot = df_plot.sort_values(['MPO', 'Year', 'variable'], ascending = [True, False, False])
print(df_plot.variable.unique())
display(df_plot.head())


df_plota = df_prod6_c.copy()
df_plot1 = df_prod6_c.copy()


df_plota = pd.melt(df_plota, id_vars = ['MPO', 'Year'])
df_plota = df_plota[df_plota['variable'].isin(['MF_total', 'SF_total'])]
fig = px.line(df_plota, x='Year', y='value', color='variable', markers=True)
fig.update_layout(title = 'MF vs SF Units (Total) SACOG')
fig.show()

df_plot1 = pd.melt(df_plot1, id_vars = ['MPO', 'Year'])
df_plot1 = df_plot1[df_plot1['variable'].isin(['MF_2to4', 'MF_5plus', 'SF_RR', 'SF_SFLL', 'SF_SFSL'])]
fig = px.line(df_plot1, x='Year', y='value', color='variable', markers=True)
fig.update_layout(title = 'MF vs SF Units (Detailed) SACOG')
fig.show()



df_plot1 = df_prod6_c.copy()

df_plot1 = pd.melt(df_plot1, id_vars = ['MPO', 'Year'])
df_plot1 = df_plot1[df_plot1['variable'].isin(['MF_total', 'SF_SFLL', 'SF_SFSL'])]

df_plot1['percentage'] = 100*df_plot1['value'] / df_plot1.groupby(['MPO', 'Year'])['value'].transform('sum')

df_plot1['variable_sort'] = pd.Categorical(df_plot1['variable'], ['MF_total', 'SF_SFSL', 'SF_SFLL'])
df_plot1 = df_plot1.sort_values(['MPO', 'Year', 'variable_sort'], ascending = [True, False, False])
df_plot1 = df_plot1.drop('variable_sort', axis = 1)

display(df_plot1.head())

fig = px.bar(df_plot1, x='Year', y='percentage', color='variable')
fig.update_layout(title = 'MF vs SF Units (Detailed) SACOG')
fig.show()



year_start = df_prod6_c['Year'].min()
year_end   = df_prod6_c['Year'].max()


if EXPORT:

    indicator = 'Production_6'
    source = 'SACOG Housing Permit Data'
    sample_type = 'SACOG Housing'
    params = {
        'sample': sample_type
        , 'indicator': indicator
        , 'start_year': year_start
        , 'end_year': year_end
        , 'estimate': None
        , 'moe_thresh': None
        , 'source': source
        , 'geo': 'Sacramento Six-County Region'
    }

    # Update overall about documentations workbook
    df_about = func.write_about(params)
    with pd.ExcelWriter(FILE_ABOUT, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_about.to_excel(writer, index=False, sheet_name=indicator, header=False)

    # Exporting
    print('Exporting...')
    print('')
    paths_out = [PATH_MAIN / 'Vibrant and Inclusive Places' / 'Development' / 'Housing Production' / indicator, PATH_SERVER]

    for path_out in paths_out:
        params['geo'] = 'Jurisdictions'
        export_housing(df_prod6_a, params, path_out)
            
        params['geo'] = 'Counties'
        export_housing(df_prod6_b, params, path_out)
        
        params['geo'] = 'MPO'
        export_housing(df_prod6_c, params, path_out)

    print('Success!!')

    




Organizing indicator Production_6 by Jurisdictions


,MPO,County,Jurisdiction,Year,SF_total,MF_total,ADU_total,SF_RR,SF_SFLL,SF_SFSL,MF_2to4,MF_5plus,ADU_RR,ADU_SFLL,ADU_SFSL
0,SACOG,EL DORADO,PLACERVILLE,2025,0,0,0,0,0,0,0,0,0,0,0
1,SACOG,EL DORADO,PLACERVILLE,2024,20,2,10,0,18,2,2,0,3,6,1
2,SACOG,EL DORADO,PLACERVILLE,2023,1,0,3,1,0,0,0,0,0,2,1
3,SACOG,EL DORADO,PLACERVILLE,2022,3,0,10,-1,4,0,0,0,3,7,0
4,SACOG,EL DORADO,PLACERVILLE,2021,8,0,7,3,5,0,0,0,0,7,0


Organizing indicator Production_6 by Counties


,MPO,County,Year,SF_total,MF_total,ADU_total,SF_RR,SF_SFLL,SF_SFSL,MF_2to4,MF_5plus,ADU_RR,ADU_SFLL,ADU_SFSL
0,SACOG,EL DORADO,2025,395,45,59,67,235,93,3,42,43,15,1
1,SACOG,EL DORADO,2024,480,72,72,91,291,98,4,68,48,21,3
2,SACOG,EL DORADO,2023,471,14,56,67,317,87,0,14,37,15,4
3,SACOG,EL DORADO,2022,595,0,57,82,413,100,0,0,44,13,0
4,SACOG,EL DORADO,2021,160,6,16,18,118,24,6,0,8,8,0


Organizing indicator Production_6 by MPO


,MPO,Year,SF_total,MF_total,ADU_total,SF_RR,SF_SFLL,SF_SFSL,MF_2to4,MF_5plus,ADU_RR,ADU_SFLL,ADU_SFSL
0,SACOG,2025,7218,4743,625,231,3562,3418,172,4571,107,389,127
1,SACOG,2024,7666,4114,758,325,3936,3405,152,3962,137,469,152
2,SACOG,2023,7528,3542,626,297,3919,3312,125,3417,119,394,113
3,SACOG,2022,8748,2616,388,341,4349,4058,44,2572,101,228,59
4,SACOG,2021,7322,2758,311,400,3384,3538,170,2588,73,203,35


<ArrowStringArray>
[ 'SF_total',   'SF_SFSL',   'SF_SFLL',     'SF_RR',  'MF_total',  'MF_5plus',
   'MF_2to4', 'ADU_total',  'ADU_SFSL',  'ADU_SFLL',    'ADU_RR']
Length: 11, dtype: str


,MPO,Year,variable,value
0,SACOG,2025,SF_total,7218
125,SACOG,2025,SF_SFSL,3418
100,SACOG,2025,SF_SFLL,3562
75,SACOG,2025,SF_RR,231
25,SACOG,2025,MF_total,4743


,MPO,Year,variable,value,percentage
100,SACOG,2025,SF_SFLL,3562,30.384714
125,SACOG,2025,SF_SFSL,3418,29.156359
25,SACOG,2025,MF_total,4743,40.458927
101,SACOG,2024,SF_SFLL,3936,34.360541
126,SACOG,2024,SF_SFSL,3405,29.725011


Exporting...

Success!!


In [ ]:


## Production_7


print('Organizing indicator Production_7 by Jurisdictions')
df_prod7_a = df_housing.copy()
df_prod7_a = df_prod7_a[['MPO', 'County', 'Jurisdiction', 'Year', 'ADU_total']]
df_prod7_a = df_prod7_a[df_prod7_a['Year'] >= 2018]
df_prod7_a = df_prod7_a.reset_index(drop=True)
display(df_prod7_a.head(5))

print('Organizing indicator Production_7 by Counties')
df_prod7_b = df_housing.copy()
df_prod7_b = df_prod7_b.groupby(['MPO', 'County', 'Year'], as_index = False, sort = False)['ADU_total'].agg('sum')
df_prod7_b = df_prod7_b[df_prod7_b['Year'] >= 2018]
df_prod7_b = df_prod7_b.reset_index(drop=True)
display(df_prod7_b.head(5))

print('Organizing indicator Production_7 by the entire SACOG Region')
df_prod7_c = df_housing.copy()
df_prod7_c = df_prod7_c.groupby(['MPO', 'Year'], as_index = False, sort = False)['ADU_total'].agg('sum')
df_prod7_c = df_prod7_c[df_prod7_c['Year'] >= 2018]
df_prod7_c = df_prod7_c.reset_index(drop=True)
display(df_prod7_c.head(5))


df_plot = df_prod7_a.copy()

df_plot = pd.melt(df_plot, id_vars = ['Year', 'County', 'Jurisdiction'])
fig = px.line(df_plot, x='Year', y='value', color='County', line_dash='Jurisdiction', markers=True)
fig.update_layout(title = 'Total Housing Permits by County/Jurisdiction')

fig.show()


df_plot = df_prod7_b.copy()

df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year', 'County'])
fig = px.line(df_plot, x='Year', y='value', color='County', markers=True)
fig.update_layout(title = 'Total Housing Permits by County')

fig.show()


df_plot = df_prod7_c.copy()

df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year']) 
fig = px.bar(df_plot, x='Year', y='value')
fig.update_layout(title = 'Total Housing Permits SACOG')

fig.show()



year_start = df_prod7_c['Year'].min()
year_end   = df_prod7_c['Year'].max()

if EXPORT:

    indicator = 'Production_7'
    source = 'SACOG Housing Permit Data'
    sample_type = 'SACOG Housing'
    params = {
        'sample': sample_type
        , 'indicator': indicator
        , 'start_year': year_start
        , 'end_year': year_end
        , 'estimate': None
        , 'moe_thresh': None
        , 'source': source
        , 'geo': 'Sacramento Six-County Region'
    }

    # Update overall about documentations workbook
    df_about = func.write_about(params)
    with pd.ExcelWriter(FILE_ABOUT, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_about.to_excel(writer, index=False, sheet_name=indicator, header=False)

    # Exporting
    print('Exporting...')
    print('')
    paths_out = [PATH_MAIN / 'Vibrant and Inclusive Places' / 'Development' / 'Housing Production' / indicator, PATH_SERVER]

    for path_out in paths_out:
        params['geo'] = 'Jurisdictions'
        export_housing(df_prod7_a, params, path_out)
            
        params['geo'] = 'Counties'
        export_housing(df_prod7_b, params, path_out)
        
        params['geo'] = 'MPO'
        export_housing(df_prod7_c, params, path_out)

    print('Success!!')

    

In [ ]:



## Location_1



gb_jurisdiction = df_housing.groupby(['MPO', 'County', 'Jurisdiction', 'Year'], sort=False, as_index=False)
gb_county       = df_housing.groupby(['MPO', 'County'                , 'Year'], sort=False, as_index=False)
gb_mpo          = df_housing.groupby(['MPO'                          , 'Year'], sort=False, as_index=False)


df_housing['COMTYP_AGNL_NA'] = df_housing['COMTYP_AGNL'] + df_housing['COMTYP_NA']

metrics = ['COMTYP_CC', 'COMTYP_EC', 'COMTYP_DC', 'COMTYP_RR', 'COMTYP_AGNL_NA']

#Jurisdiction level
print('Organizing indicator Location_1 by Jurisdictions')
df_loc1_a = gb_jurisdiction[metrics].sum() 
display(df_loc1_a.head(5))

#County level 
print('Organizing indicator Location_1 by Counties')
df_loc1_b = gb_county[metrics].sum()
display(df_loc1_b.head(5))

#MPO level
print('Organizing indicator Location_1 by MPO')
df_loc1_c = gb_mpo[metrics].sum()
display(df_loc1_c.head(5))


df_plot1 = df_loc1_a.copy()
df_plot2 = df_loc1_a.copy()
df_plot3 = df_loc1_a.copy()
df_plot4 = df_loc1_a.copy()
df_plot5 = df_loc1_a.copy()


df_plot1 = pd.melt(df_plot1, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plot1 = df_plot1[df_plot1['variable'] == 'COMTYP_CC']
fig = px.line(df_plot1, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'Community Type Commercial Corridor Units by Jurisdiction')
fig.show()

df_plot2 = pd.melt(df_plot2, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plot2 = df_plot2[df_plot2['variable'] == 'COMTYP_EC']
fig = px.line(df_plot2, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'Community Type Established Corridor Units by Jurisdiction')
fig.show()

df_plot3 = pd.melt(df_plot3, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plot3 = df_plot3[df_plot3['variable'] == 'COMTYP_DC']
fig = px.line(df_plot3, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'Community Type Developing Corridor Units by Jurisdiction')
fig.show()

df_plot4 = pd.melt(df_plot4, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plot4 = df_plot4[df_plot4['variable'] == 'COMTYP_RR']
fig = px.line(df_plot4, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'Community Type Rural Residential Units by Jurisdiction')
fig.show()

df_plot5 = pd.melt(df_plot5, id_vars = ['Year', 'County', 'Jurisdiction'])
df_plot5 = df_plot5[df_plot5['variable'] == 'COMTYP_AGNL_NA']
fig = px.line(df_plot5, x='Year', y='value', color='County', line_dash = 'Jurisdiction', markers=True)
fig.update_layout(title = 'Community Type Agriculture and Other and NA Units by Jurisdiction')
fig.show()


df_plot = df_loc1_c.copy()
df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year'])
df_plot = df_plot.sort_values(['MPO', 'Year', 'variable'], ascending = [True, False, False])
print(df_plot.variable.unique())

display(df_plot.head())


df_loc1_a = df_loc1_a[df_loc1_a['Year'] >= 2008].reset_index(drop=True)
df_loc1_b = df_loc1_b[df_loc1_b['Year'] >= 2008].reset_index(drop=True)
df_loc1_c = df_loc1_c[df_loc1_c['Year'] >= 2008].reset_index(drop=True)

year_start = df_loc1_c['Year'].min()
year_end   = df_loc1_c['Year'].max()

if EXPORT:

    indicator = "Location_1"
    source = 'SACOG Housing Permit Data'
    sample_type = 'SACOG Housing'
    params = {
        'sample': sample_type
        , 'indicator': indicator
        , 'start_year': year_start
        , 'end_year': year_end
        , 'estimate': None
        , 'moe_thresh': None
        , 'source': source
        , 'geo': 'Sacramento Six-County Region'
    }

    # Update overall about documentations workbook
    df_about = func.write_about(params)
    with pd.ExcelWriter(FILE_ABOUT, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_about.to_excel(writer, index=False, sheet_name=indicator, header=False)

    # Exporting
    print('Exporting...')
    print('')
    paths_out = [PATH_MAIN / 'Vibrant and Inclusive Places' / 'Development' / 'Housing Location' / indicator, PATH_SERVER]

    for path_out in paths_out:

        params['geo'] = 'Jurisdictions'
        export_housing(df_loc1_a, params, path_out)
            
        params['geo'] = 'Counties'
        export_housing(df_loc1_b, params, path_out)
        
        params['geo'] = 'MPO'
        export_housing(df_loc1_c, params, path_out)

    print('Success!!')


    

In [ ]:



## Location_2a



gb_jurisdiction = df_housing.groupby(['MPO', 'County', 'Jurisdiction', 'Year'], sort=False, as_index=False)
gb_county       = df_housing.groupby(['MPO', 'County'                , 'Year'], sort=False, as_index=False)
gb_mpo          = df_housing.groupby(['MPO'                          , 'Year'], sort=False, as_index=False)

metrics = ['Total', 'GRZ_TOT']

#Jurisdiction level
print('Organizing indicator Location_2a by Jurisdictions')
df_loc2a_a = gb_jurisdiction[metrics].sum()
df_loc2a_a['GRZ_SHR'] = 100*df_loc2a_a['GRZ_TOT']/df_loc2a_a['Total']
df_loc2a_a = df_loc2a_a.drop(['Total', 'GRZ_TOT'], axis = 1)
display(df_loc2a_a.head(5))

#County level 
print('Organizing indicator Location_2a by Counties')
df_loc2a_b = gb_county[metrics].sum()
df_loc2a_b['GRZ_SHR'] = 100*df_loc2a_b['GRZ_TOT']/df_loc2a_b['Total']
df_loc2a_b = df_loc2a_b.drop(['Total', 'GRZ_TOT'], axis = 1)
display(df_loc2a_b.head(5))

#MPO level
print('Organizing indicator Location_2a by MPO')
df_loc2a_c = gb_mpo[metrics].sum()
df_loc2a_c['GRZ_SHR'] = 100*df_loc2a_c['GRZ_TOT']/df_loc2a_c['Total']
df_loc2a_c = df_loc2a_c.drop(['Total', 'GRZ_TOT'], axis = 1)
display(df_loc2a_c.head(5))


df_plot = df_loc2a_a.copy()

df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year', 'County', 'Jurisdiction'])
fig = px.line(df_plot, x='Year', y='value', color='County', line_dash='Jurisdiction', markers=True)
fig.update_layout(title = 'Housing Permit Green Zone Proportion by Jurisdiction')

fig.show()


df_plot = df_loc2a_b.copy()


display(df_loc2a_b.head())


df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year', 'County'])
fig = px.line(df_plot, x='Year', y='value', color='County', markers=True)
fig.update_layout(title = 'Housing Permit Green Zone Proportion by County')

fig.show()


df_plot = df_loc2a_c.copy()

display(df_loc2a_c.head())


df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year']) 
fig = px.line(df_plot, x='Year', y='value', markers=True)
fig.update_layout(title = 'Housing Permit Green Zone Proportion SACOG')

fig.show()



year_start = df_loc2a_c['Year'].min()
year_end   = df_loc2a_c['Year'].max()

if EXPORT:
    indicator = "Location_2a"
    source = 'SACOG Housing Permit Data'
    sample_type = 'SACOG Housing'
    params = {
        'sample': sample_type
        , 'indicator': indicator
        , 'start_year': year_start
        , 'end_year': year_end
        , 'estimate': None
        , 'moe_thresh': None
        , 'source': source
        , 'geo': 'Sacramento Six-County Region'
    }

    # Update overall about documentations workbook
    df_about = func.write_about(params)
    with pd.ExcelWriter(FILE_ABOUT, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_about.to_excel(writer, index=False, sheet_name=indicator, header=False)

    # Exporting
    print('Exporting...')
    print('')
    paths_out = [PATH_MAIN / 'Vibrant and Inclusive Places' / 'Development' / 'Housing Location' / indicator, PATH_SERVER]

    for path_out in paths_out:
        params['geo'] = 'Jurisdictions'
        export_housing(df_loc2a_a, params, path_out)
            
        params['geo'] = 'Counties'
        export_housing(df_loc2a_b, params, path_out)
        
        params['geo'] = 'MPO'
        export_housing(df_loc2a_c, params, path_out)

    print('Success!!')




In [ ]:


## Location_2b



gb_jurisdiction = df_housing.groupby(['MPO', 'County', 'Jurisdiction', 'Year'], sort=False, as_index=False)
gb_county       = df_housing.groupby(['MPO', 'County'                , 'Year'], sort=False, as_index=False)
gb_mpo          = df_housing.groupby(['MPO'                          , 'Year'], sort=False, as_index=False)


df_housing['GRZ_MF'   ] = df_housing['GRZ_MF2t4'   ] + df_housing['GRZ_MF5'   ]
df_housing['GRZ_MF_NO'] = df_housing['GRZ_MF2t4_NO'] + df_housing['GRZ_MF5_NO']

metrics = ['GRZ_SF', 'GRZ_MF', 'GRZ_MF2t4', 'GRZ_MF5', 'GRZ_SFLL', 'GRZ_SFSL', 'GRZ_SFSL_NO', 'GRZ_RR_SFLL_NO', 'GRZ_MF_NO']


#Jurisdiction level for Green Zone region 
print('Organizing indicator Location_2a by Jurisdictions for Green Zone region')
df_loc2b_a = gb_jurisdiction[metrics].sum()
display(df_loc2b_a.head(5))

#County level for Green Zone Region
print('Organizing indicator Location_2a by Counties for Green Zone region')
df_loc2b_b = gb_county[metrics].sum()
display(df_loc2b_b.head(5))

#MPO level for Green Zone Region 
print('Organizing indicator Location_2a by MPO for Green Zone region')
df_loc2b_c = gb_mpo[metrics].sum()
display(df_loc2b_c.head(5))


df_plot = df_loc2b_c.copy()
df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Year'])
df_plot = df_plot.sort_values(['MPO', 'Year', 'variable'], ascending = [True, False, False])
print(df_plot.variable.unique())
df_plot.columns = [col.lower() for col in df_plot.columns]
df_plot = df_plot[~df_plot['variable'].isin(['GRZ_SF', 'GRZ_MF2t4', 'GRZ_MF5'])]
conditions = [
    df_plot['variable'].isin(['GRZ_SFSL_NO', 'GRZ_RR_SFLL_NO', 'GRZ_MF_NO'])
    , df_plot['variable'].isin(['GRZ_SFSL', 'GRZ_SFLL', 'GRZ_MF'])
]
choices = ['Not in Green Zone', 'Green Zone']
df_plot['green_zone'] = np.select(conditions, choices, default = 'No')

conditions = [
    df_plot['variable'].isin(['GRZ_SFSL_NO', 'GRZ_SFSL'])
    , df_plot['variable'].isin(['GRZ_RR_SFLL_NO', 'GRZ_SFLL'])
    , df_plot['variable'].isin(['GRZ_MF_NO', 'GRZ_MF'])
]
choices = ['GRZ_SFSL', 'GRZ_SFLL', 'GRZ_MF']
df_plot['product_type'] = np.select(conditions, choices, default = 'No')
df_plot = df_plot[['mpo', 'year', 'green_zone', 'product_type', 'value']]
df_plot = df_plot.groupby(['mpo', 'year', 'green_zone', 'product_type'], as_index = False)['value'].sum()
df_plot = df_plot.sort_values(['mpo', 'year', 'green_zone', 'product_type'], ascending = [True, False, True, True])
display(df_plot.head())



year_start = df_loc2b_c['Year'].min()
year_end   = df_loc2b_c['Year'].max()

if EXPORT:
    indicator = "Location_2b"
    source = 'SACOG Housing Permit Data'
    sample_type = 'SACOG Housing'
    params = {
        'sample': sample_type
        , 'indicator': indicator
        , 'start_year': year_start
        , 'end_year': year_end
        , 'estimate': None
        , 'moe_thresh': None
        , 'source': source
        , 'geo': 'Sacramento Six-County Region'
    }

    # Update overall about documentations workbook
    df_about = func.write_about(params)
    with pd.ExcelWriter(FILE_ABOUT, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_about.to_excel(writer, index=False, sheet_name=indicator, header=False)

    # Exporting
    print('Exporting...')
    print('')
    paths_out = [PATH_MAIN / 'Vibrant and Inclusive Places' / 'Development' / 'Housing Location' / indicator, PATH_SERVER]

    for path_out in paths_out:
        params['geo'] = 'Jurisdictions'
        export_housing(df_loc2b_a, params, path_out)
            
        params['geo'] = 'Counties'
        export_housing(df_loc2b_b, params, path_out)
        
        params['geo'] = 'MPO'
        export_housing(df_loc2b_c, params, path_out)

    print('Success!!')

    


***

Policy_5

***

In [ ]:



# indicator = 'Policy_5'
# path_proj = path_main / 'Vibrant and Inclusive Places' / 'Development' / 'Policy and Planning' / 'Policy_5'

# file_proj = path_proj / 'projection (for policy_5).xlsx'

# df_proj = pd.read_excel(file_proj)
# display(df_proj.head())
# display(df_housing.head())



# df_policy5 = df_prod6_c.copy()

# metrics = ['SF_RR', 'SF_SFLL', 'SF_SFSL', 'MF_total']

# df_policy5 = df_prod6_c[['MPO', 'Year'] + metrics]
# df_policy5 = df_policy5.sort_values(['MPO', 'Year'])

# conditions = [   
#          df_policy5['Year'].isin(sequence(2000, 2007, 1))
#        , df_policy5['Year'].isin(sequence(2008, 2011, 1))
#        , df_policy5['Year'].isin(sequence(2012, 2015, 1))
#        , df_policy5['Year'].isin(sequence(2016, 2019, 1))
#        , df_policy5['Year'].isin(sequence(2020, 2023, 1))
#              ]
# choices = ["2001-2007", "2008-2011", "2012-2015", "2016-2019", "2020-2023"]
# df_policy5["Period"] = np.select(conditions, choices)

# display(df_policy5.head())



# df_plot = df_policy5.copy()

# df_plot = df_plot.groupby(['MPO', 'Period'], as_index = False)[metrics].mean()
# df_plot[metrics] = df_plot[metrics].apply(round)
# list_data = [['SACOG', '2024-2040', 110, 600, 2100, 5700]]
# df_forecast = pd.DataFrame(list_data, columns = ['MPO', 'Period', 'SF_RR', 'SF_SFLL', 'SF_SFSL', 'MF_total'])
# df_plot = pd.concat([df_plot, df_forecast])
# df_plot = df_plot.rename(columns = {'SF_RR':'Rural Residential', 'SF_SFLL':'Single Family-Large Lot'
#                                     , 'SF_SFSL':'Single Family-Small Lot', 'MF_total':'Attached'})

# df_plot = pd.melt(df_plot, id_vars = ['MPO', 'Period'])
# df_plot['variable_sort'] = pd.Categorical(df_plot['variable'], ['Rural Residential', 'Single Family-Large Lot', 'Single Family-Small Lot', 'Attached'])
# df_plot = df_plot.sort_values(['MPO', 'Period', 'variable_sort'], ascending = [True, True, False])
# df_plot = df_plot.drop('variable_sort', axis = 1)

# fig = px.bar(df_plot, x = 'Period', y = 'value', color = 'variable')

# fig.show()


# list_df_forecast = []

# for year in range(2024, 2040):

#     annual_growth = [110, 600, 2100, 5700]
    
#     list_data = [['SACOG', '2024-2040', year] + annual_growth]
#     df_forecast = pd.DataFrame(list_data, columns = ['MPO', 'Period', 'Year', 'SF_RR', 'SF_SFLL', 'SF_SFSL', 'MF_total'])

#     list_df_forecast.append(df_forecast)

# df_forecast = pd.concat(list_df_forecast)
# df_policy5 = pd.concat([df_policy5, df_forecast])
# df_policy5 = df_policy5.set_index(['MPO', 'Period', 'Year']).reset_index()

# display(df_policy5.head())

